### KNN Classifier Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from scipy.stats.mstats import winsorize
from tensorflow import keras

# load dataset
df = pd.read_csv('data/DSI_kickstarterscrape_dataset.csv', encoding='ISO-8859-1')
df = df.dropna()
df = df.query("location.str.match(r'.*, [A-Z]{2}$')", engine='python')  # keep US projects only

# extract the state abbreviation (last two capital letters after comma)
df['state'] = df['location'].str.extract(r", ([A-Z]{2})$")

# drop columns that aren't useful for modeling
drop_cols = ['name', 'url', 'project id', 'location', 'reward levels']
df = df.drop(columns=drop_cols, errors='ignore')

# convert categorical columns to dummy variables
df = pd.get_dummies(df, columns=['category', 'subcategory', 'state'], drop_first=True, dtype=int)

# extract funded month from date
df['funded_month'] = pd.to_datetime(df['funded date'], errors='coerce').dt.month
df.drop(columns=['funded date'], inplace=True, errors='ignore')

# winsorize numerical columns to limit outliers
for c in ['goal', 'pledged', 'backers', 'duration']:
df[c] = winsorize(df[c], limits=[0.01, 0.01])

# define target and features
df['success'] = (df['pledged'] >= df['goal']).astype(int)
y = df['success']
X = df.drop(columns=['status', 'pledged', 'backers', 'success', 'funded percentage', 'updates'], errors='ignore')

# split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# tune knn with gridsearchcv
param_grid = {
    'n_neighbors': [5, 10, 15, 25],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
scoring = {'accuracy': 'accuracy', 'f1_macro': make_scorer(f1_score, average='macro')}

grid = GridSearchCV(KNeighborsClassifier(), param_grid, scoring=scoring, refit='f1_macro', cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

# evaluate results
print("best parameters:", grid.best_params_)
print(f"best cv f1-score: {grid.best_score_:.3f}")

y_pred = grid.best_estimator_.predict(X_test)
print(f"test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"test f1-score: {f1_score(y_test, y_pred, average='macro'):.3f}")


best parameters: {'n_neighbors': 25, 'p': 1, 'weights': 'uniform'}
best cv f1-score: 0.635
test accuracy: 0.636
test f1-score: 0.635


### Neural Network

In [ ]:
print("\ntraining neural network")

# define model
nn_model = keras.Sequential([
    keras.layers.Dense(128, input_dim=X_train.shape[1], activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

# compile
nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
nn_model.summary()

# train
history = nn_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

# test performance
loss, accuracy = nn_model.evaluate(X_test, y_test)
print(f"\ntest accuracy: {accuracy * 100:.2f}%")



training neural network...


/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-10-24 20:08:25.792244: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        15,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,553 (92.00 KB)

 Trainable params: 23,553 (92.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.6342 - loss: 0.6453 - val_accuracy: 0.6789 - val_loss: 0.5963
Epoch 2/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6728 - loss: 0.6011 - val_accuracy: 0.6786 - val_loss: 0.5871
Epoch 3/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6829 - loss: 0.5872 - val_accuracy: 0.6860 - val_loss: 0.5819
Epoch 4/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6943 - loss: 0.5786 - val_accuracy: 0.6843 - val_loss: 0.5760
Epoch 5/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6944 - loss: 0.5736 - val_accuracy: 0.6885 - val_loss: 0.5743
Epoch 6/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7010 - loss: 0.5677 - val_accuracy: 0.6955 - val_loss: 0.5682
Epoch 7/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7033 - loss: 0.5589 - val_accuracy: 0.7012 - val_loss: 0.5639
Epoch 8/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7094 - loss: 0.5544 - val_accuracy: 0.